In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
from ml_experiments.analyze import get_df_runs_from_mlflow_sql, get_missing_entries
from pathlib import Path
import os
from functools import partial

# Save Results

## Load mlflow runs

In [ ]:
results_dir = Path.cwd().parent / "results" / "real"
os.makedirs(results_dir, exist_ok=True)

In [ ]:
db_port = 5001
db_name = "cohirf"
url = f"postgresql://user@localhost:{db_port}/{db_name}"
engine = create_engine(url)
query = "SELECT experiments.name from experiments"
experiment_names = pd.read_sql(query, engine)["name"].tolist()
experiments_names = [exp for exp in experiment_names if (exp.startswith("real-"))]

In [ ]:
experiments_names

In [ ]:
query = "SELECT DISTINCT(key) FROM params WHERE key LIKE 'best/%%'"
best_params = pd.read_sql(query, engine)["key"].tolist()

In [ ]:
params_columns = [
    "model",
    "dataset_id",
    "n_trials",
    "dataset_name",
    "standardize",
    "hpo_metric",
    "direction",
    "hpo_seed",
    "cohirf_kwargs/consensus_strategy",
    "consensus_strategy",
    "batch_sample_strategy",
    "cohirf_kwargs/n_samples_representative",
    "n_samples_representative",
	"seed_model",
	"model_params/n_samples_representative",
] + best_params

In [ ]:
latest_metrics_columns = [
    "fit_model_return_elapsed_time",
    "max_memory_used_after_fit",
    "max_memory_used",
	"best/n_clusters_",
    "best/rand_score",
    "best/adjusted_rand",
    "best/mutual_info",
    "best/adjusted_mutual_info",
    "best/normalized_mutual_info",
    "best/homogeneity_completeness_v_measure",
    "best/silhouette",
    "best/calinski_harabasz_score",
    "best/davies_bouldin_score",
    "best/inertia_score",
    "best/homogeneity",
    "best/completeness",
    "best/v_measure",
    "best/elapsed_time",
]

In [ ]:
tags_columns = ["raised_exception", "EXCEPTION", "mlflow.parentRunId", "Last step finished"]

In [ ]:
runs_columns = ['run_uuid', 'status', 'start_time', 'end_time']
experiments_columns = []
other_table = 'params'
other_table_keys = params_columns

In [ ]:
df_params = get_df_runs_from_mlflow_sql(engine, runs_columns=runs_columns, experiments_columns=experiments_columns, experiments_names=experiments_names, other_table=other_table, other_table_keys=other_table_keys)
df_latest_metrics = get_df_runs_from_mlflow_sql(engine, runs_columns=['run_uuid'], experiments_columns=experiments_columns, experiments_names=experiments_names, other_table='latest_metrics', other_table_keys=latest_metrics_columns)
df_tags = get_df_runs_from_mlflow_sql(engine, runs_columns=['run_uuid'], experiments_columns=experiments_columns, experiments_names=experiments_names, other_table='tags', other_table_keys=tags_columns)

In [ ]:
dataset_characteristics = pd.read_csv(results_dir / "datasets_characteristics.csv", index_col=0)
dataset_characteristics.index = dataset_characteristics["openml_id"].astype(str)

In [ ]:
df_runs_raw = df_params.join(df_latest_metrics)
df_runs_raw = df_runs_raw.join(df_tags)
df_runs_raw["start_time"] = pd.to_datetime(df_runs_raw["start_time"], unit="ms")
df_runs_raw = df_runs_raw.loc[df_runs_raw["start_time"] > "2025-11-01"]  # to filter out old runs

In [ ]:
df_runs_raw = df_runs_raw.join(dataset_characteristics, on="dataset_id", rsuffix="_dataset")
df_runs_raw.to_csv(results_dir / 'df_runs_raw_cer.csv', index=True)

In [ ]:
df_runs_raw["n_trials"].unique()

In [ ]:
df_runs_raw

In [ ]:
df_runs_raw = pd.read_csv(results_dir / "df_runs_raw_cer.csv", index_col=0, low_memory=False)
df_runs_raw = df_runs_raw.dropna(subset=["n_trials"])
df_runs_raw["model"] = df_runs_raw["model"] + "-" + df_runs_raw["n_trials"].astype(int).astype(str)
df_runs_raw_parents = df_runs_raw.copy()
df_runs_raw_parents = df_runs_raw_parents.loc[df_runs_raw_parents["mlflow.parentRunId"].isna()]

In [ ]:
df_runs_raw_parents.head(5)

## Delete duplicate runs (if any) and complete some models that cannot run with some datasets

In [ ]:
non_duplicate_columns = [
    "model",
    "dataset_id",
	"standardize",
	"hpo_metric",
	"hpo_seed",
	"model_params/n_samples_representative",
]
# df_runs_parents.loc[df_runs_parents["best/n_clusters_"]*0.5 > df_runs_parents["n_instances"], "best/adjusted_rand"] = 
df_runs_parents = df_runs_raw_parents.dropna(axis=0, how="all", subset=["best/adjusted_rand"]).copy()
# add back runs that were not evaluated because we judged too many clusters (but they run anyway)
# df_valid_runs = df_runs_raw_parents.loc[df_runs_raw_parents["best/n_clusters_"] > df_runs_raw_parents["n_instances"]*0.5].copy()
# df_runs_parents = pd.concat([df_runs_parents, df_valid_runs], axis=0)
df_runs_parents = df_runs_parents.loc[(~df_runs_parents.duplicated(non_duplicate_columns))]
# fill missing values with "None"
df_runs_parents = df_runs_parents.fillna("None")

In [ ]:
# get number of children runs that raised exception for each parent run
children_exceptions = df_runs_raw.groupby("mlflow.parentRunId")["raised_exception"].sum()
df_runs_parents["n_children_raised_exception"] = df_runs_parents.index.map(children_exceptions).fillna(0)

In [ ]:
df_runs_parents.loc[(df_runs_parents["n_children_raised_exception"] > 0) & (df_runs_parents["raised_exception"] == False), ["dataset_id", "model", "hpo_metric", "n_children_raised_exception"]]

In [ ]:
df_to_cat = []
hpo_metrics = [
    "adjusted_rand",
    "adjusted_mutual_info",
    "calinski_harabasz_score",
    "silhouette",
    "davies_bouldin_score",
    "normalized_mutual_info",
]
standardize = [True]
hpo_seed = [i for i in range(5)]
fill_value = pd.NA
fill_columns = ["best/adjusted_rand", "best/adjusted_mutual_info", "best/calinski_harabasz_score", "best/silhouette", "best/davies_bouldin_score", "best/normalized_mutual_info"]

In [ ]:
# Too memory intensive
dataset_ids_to_complete = [182, 554, 1478, 1568, 40685]
model_names = [
    "CoHiRF-SC-SRGF-60",
    "CoHiRF-SC-SRGF-top-down-60",
    "CoHiRF-SC-SRGF-top-down-inv-60",
    "SpectralSubspaceRandomization-60",
    "CoHiRF-SC-SRGF-1R-60",
    "CoHiRF-SC-SRGF-top-down-1R-60",
    "CoHiRF-SC-SRGF-top-down-inv-1R-60",
    "CoHiRF-SC-SRGF-2R-60",
    "CoHiRF-SC-SRGF-top-down-2R-60",
	"CoHiRF-SC-SRGF-top-down-inv-2R-60",
]
for dataset_id in dataset_ids_to_complete:
    for model_name in model_names:
        for hpo_metric in hpo_metrics:
            for std in standardize:
                for seed in hpo_seed:
                    new_row = {
						"dataset_id": dataset_id,
						"model": model_name,
						"hpo_metric": hpo_metric,
						"standardize": std,
						"hpo_seed": seed
					}
                    for col in fill_columns:
                        new_row[col] = fill_value
                    df_to_cat.append(new_row)

In [ ]:
df_runs_parents = pd.concat([df_runs_parents, pd.DataFrame(df_to_cat)], axis=0)

# Missing

In [ ]:
model_nickname = df_runs_parents['model'].unique().tolist()
model_nickname.sort()
model_nickname

In [ ]:
non_duplicate_columns = [
	"model",
	"dataset_id",
	"standardize",
	"hpo_metric",
	"hpo_seed",
]

In [ ]:
model_nickname = [
    "CoHiRF-60",
    "CoHiRF-top-down-60",
    "CoHiRF-1000-60",
    "CoHiRF-DBSCAN-60",
    "CoHiRF-DBSCAN-top-down-60",
    "CoHiRF-KernelRBF-60",
	"CoHiRF-KernelRBF-top-down-60",
    "CoHiRF-SC-SRGF-1R-60",
	"CoHiRF-SC-SRGF-2R-60",
	"CoHiRF-SC-SRGF-60",
    "CoHiRF-SC-SRGF-top-down-60",
    "DBSCAN-60",
    "KMeans-60",
    "KernelRBFKMeans-60",
    "SpectralSubspaceRandomization-60",
]
dataset_id = [
    61,
    46773,
    46776,
    46778,
    46779,
    46782,
    46783,
]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
model_nickname = [
    "CoHiRF-1R-60",
    "CoHiRF-2R-60",
    "CoHiRF-DBSCAN-1R-60",
    "CoHiRF-DBSCAN-2R-60",
    "CoHiRF-KernelRBF-1R-60",
    "CoHiRF-KernelRBF-2R-60",
    "CoHiRF-SC-SRGF-1R-60",
    "CoHiRF-SC-SRGF-2R-60",
]
dataset_id = [
    46773,
	46779,
	1568,
	40685,
    46783,
]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
df = df_runs_parents.copy()
df = df.loc[df["model"] == "CoHiRF-60"]
df = df.loc[df["dataset_id"] == 40685]
df[["dataset_id", "model", "max_memory_used"]]

In [ ]:
model_nickname = [
    "BatchCoHiRF-DBSCAN-1R-1iter-random-60",
    "BatchCoHiRF-DBSCAN-2R-1iter-random-60",
    "BatchCoHiRF-DBSCAN-1R-1iter-random-60",
    "BatchCoHiRF-DBSCAN-2R-1iter-random-60",
    "CoHiRF-DBSCAN-60",
    "CoHiRF-DBSCAN-top-down-60",
]
dataset_id = [
    40685,
]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
model_nickname = [
    "BatchCoHiRF-SC-SRGF-1R-1iter-random-60",
    "BatchCoHiRF-SC-SRGF-2R-1iter-random-60",
	"BatchCoHiRF-SC-SRGF-1iter-random-60",
	"BatchCoHiRF-SC-SRGF-top-down-1iter-random-60",
]
dataset_id = [
	40685,
    1568,
]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
model_nickname = [
    "BatchCoHiRF-SC-SRGF2-1R-1iter-random-60",
    "BatchCoHiRF-SC-SRGF2-2R-1iter-random-60",
    "BatchCoHiRF-SC-SRGF2-1iter-random-60",
    "BatchCoHiRF-SC-SRGF2-top-down-1iter-random-60",
]
dataset_id = [
    40685,
    # 1568,
]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
model_nickname = [
    "CoHiRF-SC-SRGF2-1R-60",
    "CoHiRF-SC-SRGF2-2R-60",
	"CoHiRF-SC-SRGF2-60",
	"CoHiRF-SC-SRGF2-top-down-60",
]
dataset_id = [46783]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
# run with smaller grid for datasets that have less clusters
model_nickname = ["KMeans2-60", "KernelRBFKMeans2-60","SpectralSubspaceRandomization2-60"]
dataset_id = [
    46773,
    46779,
    1568,
]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
model_nickname = [
    "CoHiRF2-1R-60",
    "CoHiRF2-2R-60",
	"CoHiRF2-60",
	"CoHiRF2-top-down-60",
    "CoHiRF-KernelRBF2-1R-60",
    "CoHiRF-KernelRBF2-2R-60",
	"CoHiRF-KernelRBF2-60",
	"CoHiRF-KernelRBF2-top-down-60",
]
dataset_id = [46783, 40685]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
non_duplicate_columns = [
    "model",
    "dataset_id",
    "standardize",
    "hpo_metric",
    "hpo_seed",
    "model_params/n_samples_representative",
]

In [ ]:
model_nickname = ["CoHiRF-60", "CoHiRF2-60"]
dataset_id = [40685]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
n_samples_representative = [100, 1000, 5000, 10000, 20000]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed, n_samples_representative]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
df = df_runs_raw_parents.copy()
df = df.loc[df["model"] == "CoHiRF-60"]
df = df.loc[df["dataset_id"] == 40685]
df = df.loc[df["hpo_metric"] == "adjusted_rand"]
df = df.loc[df["hpo_seed"] == 0]
df[["dataset_id", "model", "hpo_metric", "hpo_seed", "n_samples_representative", "best/adjusted_rand"]]

In [ ]:
model_nickname = [
    "CoHiRF-SC-SRGF-1R-60",
    "CoHiRF-SC-SRGF-2R-60",
    "SpectralSubspaceRandomization-60",
]
dataset_id = [
    163,
    477,
    10,
    61,
    48,
    46336,
    46331,
    46334,
    7,
    51,
    49,
    39,
    46335,
    42855,
    35,
    46333,
    40496,
    478,
    377,
    11,
    42,
    188,
    469,
    458,
    54,
    46332,
    307,
    40966,
    1468,
    40733,
    23,
    1501,
    1493,
    40975,
    40982,
    18,
    22,
    16,
    14,
    12,
    40979,
    1466,
    40984,
    23380,
    40670,
    46,
    1497,
    30,
    40499,
    28,
    1475,
    182,
    300,
    41164,
    4538,
    375,
]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
model_nickname = [
    "BatchCoHiRF-1iter-random-60",
    "BatchCoHiRF-1iter-random-top-down-60",
    "BatchCoHiRF-DBSCAN-1iter-random-60",
    "BatchCoHiRF-DBSCAN-1iter-random-top-down-60",
    "BatchCoHiRF-KernelRBF-1iter-random-60",
    "BatchCoHiRF-KernelRBF-1iter-random-top-down-60",
    "BatchCoHiRF-SC-SRGF-1R-1iter-random-60",
    # "BatchCoHiRF-SC-SRGF-2R-1iter-random-60",
    "CoHiRF-60",
    "CoHiRF-top-down-60",
    "CoHiRF-1000-60",
    "CoHiRF-DBSCAN-60",
    "CoHiRF-DBSCAN-top-down-60",
    "CoHiRF-KernelRBF-60",
    "CoHiRF-KernelRBF-top-down-60",
    "CoHiRF-SC-SRGF-1R-60",
    "CoHiRF-SC-SRGF-2R-60",
    "DBSCAN-60",
    "KMeans-60",
    "KernelRBFKMeans-60",
    "SpectralSubspaceRandomization-60",
]
dataset_id = [
    554,
	40685,
	1568,
]
standardize = [True]
hpo_metric = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]
hpo_seed = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, dataset_id, standardize, hpo_metric, hpo_seed]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
df_missing

In [ ]:
# Join df_runs_raw_parents into df_missing using non_duplicate_columns to get the EXCEPTION column
df_missing_with_exception = df_missing.merge(
    df_runs_raw_parents[non_duplicate_columns + ["raised_exception", "EXCEPTION", "Last step finished"]],
    how="left",
    left_on=["model", "dataset_id", "standardize", "hpo_metric", "hpo_seed"],
    right_on=["model", "dataset_id", "standardize", "hpo_metric", "hpo_seed"],
)
df_missing_with_exception[
    [
        "model",
        "dataset_id",
        "standardize",
        "hpo_metric",
        "hpo_seed",
        "raised_exception",
        "EXCEPTION",
        "Last step finished",
    ]
]

In [ ]:
df_missing_dict = df_missing.copy()
# get only rows from high_mem_tuples
# df_missing_dict = df_missing_dict.merge(high_mem_tuples, on=["model", "dataset_id"], how="left", indicator=True)
# df_missing_dict = df_missing_dict[df_missing_dict["_merge"] == "both"].drop(columns="_merge")
# exclude rows that are in missing_ari_tuples
# df_missing_dict = df_missing_dict.merge(
# 	missing_ari_tuples, on=["model", "dataset_id"], how="left", indicator=True
# )|
# df_missing_dict = df_missing_dict[df_missing_dict["_merge"] == "left_only"].drop(columns="_merge")
# exclude rows that are in high_mem_tuples
# df_missing_dict = df_missing_dict.merge(
# 	high_mem_tuples, on=["model", "dataset_id"], how="left", indicator=True
# )
# df_missing_dict = df_missing_dict[df_missing_dict["_merge"] == "left_only"].drop(columns="_merge")
# to_drop = pd.concat([missing_ari_tuples, high_mem_tuples], ignore_index=True)
# df_missing_dict = df_missing_dict[df_missing_dict["_merge"] == "left_only"].drop(columns="_merge")

In [ ]:
# get rid of -60
df_missing_dict["model"] = df_missing_dict["model"].str.replace("-60", "")
df_missing_dict["seed_dataset_order"] = df_missing_dict["hpo_seed"]
# df_missing_dict = df_missing_dict.loc[~df_missing_dict["dataset_id"].isin([40685, 554])]
df_missing_dict.to_csv(results_dir / "df_missing_dict.csv", index=False)

In [ ]:
df_missing_dict

# Tables

In [ ]:
def get_parameters_string(row):
    parameter_names = {
		"best/alpha": "\\alpha",
		"best/avg_dims": "d",
		"best/base_model_kwargs/eps": "\\epsilon",
		"best/base_model_kwargs/min_samples": "n_{\\text{min}}",
		"best/base_model_kwargs/n_clusters": "C",
		"best/c": "c",
		"best/cohirf_kwargs/base_model_kwargs/eps": "\\epsilon",
		"best/cohirf_kwargs/base_model_kwargs/min_samples": "n_{\\text{min}}",
		"best/cohirf_kwargs/kmeans_n_clusters": "C",
		"best/cohirf_kwargs/n_features": "q",
		"best/cohirf_kwargs/repetitions": "R",
		"best/damping": "\\lambda",
		# "best/density_threshold": "\\tau",
		"best/eps": "\\epsilon",
		"best/kmeans_n_clusters": "C",
		"best/lambda_": "\\lambda",
		"best/min_bin_freq": "bin_{\\text{min}}",
		"best/min_cluster_size": "C_{\\text{min}}",
		"best/min_samples": "n_{\\text{min}}",
		"best/n_clusters": "C",
		"best/n_features": "q",
		# "best/n_partitions": "P",
		"best/n_similarities": "m",
		"best/p": "p",
		"best/repetitions": "R",
		"best/sampling_ratio": "r",
		"best/sc_n_clusters": "C",
		"best/transform_kwargs/gamma": "\\gamma",
	}
    first = True
    str = ""
    for p in parameter_names.keys():
        if not pd.isna(row[p]) and row[p] != "None":
            if not first:
                str += "; "
            else:
                first = False
            value = float(row[p])
            if value.is_integer():
                value = int(value)
                str += f"${parameter_names[p]}={value}$"
            else:
                str += f"${parameter_names[p]}={value:0.2f}$"
    return str

In [ ]:
def highlight_max(df, column_name, level=0):
    df_column = df[column_name]
    max_values = df_column.groupby(level=level).transform('max')
    is_highlighted = df_column.round(3) == max_values.round(3)
    df_css = df.copy().astype(str)
    df_css.loc[:, :] = ''
    df_css[is_highlighted] = 'font-weight: bold'
    return df_css

In [ ]:
def highlight_min(df, column_name, level=0):
    df_column = df[column_name]
    min_values = df_column.groupby(level=level).transform("min")
    is_highlighted = df_column.round(3) == min_values.round(3)
    df_css = df.copy().astype(str)
    df_css.loc[:, :] = ""
    df_css[is_highlighted] = "font-weight: bold"
    return df_css

In [ ]:
def highlight_max_index(series_index, df_column, level=0):
    max_values = df_column.groupby(level=level).transform('max')
    is_highlighted = df_column.round(3) == max_values.round(3)
    series_css = series_index.copy().astype(str)
    series_css[:] = ''
    series_css[is_highlighted.values] = 'font-weight: bold'
    return series_css

In [ ]:
def underline_2nd_max(df, column_name, level=0):
    df_column = df[column_name]
    # get the second max value
    second_max_values = df_column.groupby(level=level).transform(lambda x: x.round(3).drop_duplicates().nlargest(2).iloc[-1])
    is_underlined = df_column.round(3) == second_max_values.round(3)
    df_css = df.copy().astype(str)
    df_css.loc[:, :] = ''
    df_css[is_underlined] = 'underline: --latex--rwrap'
    return df_css

In [ ]:
def underline_2nd_min(df, column_name, level=0):
    df_column = df[column_name]
    # get the second min value
    second_min_values = df_column.groupby(level=level).transform(
        lambda x: x.round(3).drop_duplicates().nsmallest(2).iloc[-1]
    )
    is_underlined = df_column.round(3) == second_min_values.round(3)
    df_css = df.copy().astype(str)
    df_css.loc[:, :] = ""
    df_css[is_underlined] = "underline: --latex--rwrap"
    return df_css

In [ ]:
def underline_2nd_max_index(series_index, df_column, level=0):
    # get the second max value
    second_max_values = df_column.groupby(level=level).transform(lambda x: x.nlargest(2).iloc[-1])
    is_underlined = df_column.round(3) == second_max_values.round(3)
    series_css = series_index.copy().astype(str)
    series_css.loc[:] = ''
    series_css[is_underlined.values] = 'underline: --latex--rwrap'
    return series_css

In [ ]:
def get_df_metrics(df, hpo_metrics, hpo_metrics_rename):
    dfs_metrics = {}

    for hpo_metric, hpo_metric_rename in zip(hpo_metrics, hpo_metrics_rename):
        if hpo_metric.find("_rescaled") != -1:
            original_metric = hpo_metric.replace("_rescaled", "")
        else:
            original_metric = hpo_metric
        df_metric = df.loc[df["hpo_metric"] == original_metric][
            ["dataset_name", "model", "hpo_seed", f"best/{hpo_metric}"]
        ].rename(columns={f"best/{hpo_metric}": hpo_metric_rename})
        df_metric = df_metric.dropna(subset=[hpo_metric_rename])
        df_metric = df_metric.set_index(["dataset_name", "model", "hpo_seed"])
        df_metric = df_metric.astype({hpo_metric_rename: float})
        dfs_metrics[hpo_metric_rename] = df_metric

    df_metrics = pd.concat(dfs_metrics.values(), axis=1, join="outer")
    df_metrics = df_metrics.reset_index()

    # calculate mean and std
    df_metrics = df_metrics.groupby(["dataset_name", "model"]).agg(["mean", "std"])
    # flatten multiindex columns
    df_metrics.columns = [" ".join(col).strip() for col in df_metrics.columns.values]
    # drop hpo_seed level
    df_metrics = df_metrics.drop(columns=["hpo_seed mean", "hpo_seed std"])
    # Rename index levels
    df_metrics.index.names = ["Dataset", "Model"]

    # create a composite metric as the average of the metrics
    # df_metrics["Composite Metric mean"] = df_metrics[
    #     [f"{metric} mean" for metric in hpo_metrics_rename if "Rescaled" in metric]
    # ].mean(axis=1)
    # df_metrics["Composite Metric std"] = (
    #     1
    #     / len(hpo_metrics_rename)
    #     * (df_metrics[[f"{metric} std" for metric in hpo_metrics_rename if "Rescaled" in metric]] ** 2).sum(axis=1) ** 0.5
    # )
    # hpo_metrics_rename.append("Composite Metric")

    for metric in hpo_metrics_rename:
        df_metrics[f"{metric}"] = (
            df_metrics[f"{metric} mean"].apply(lambda x: f"{x:.3f}" if not pd.isna(x) else "No Run")
            + " $\\pm$ "
            + df_metrics[f"{metric} std"].apply(lambda x: f"{x:.3f}" if not pd.isna(x) else "No Run")
        )
    # Calculate mean and std times for each dataset-model combination across all metrics
    df_times = (
        df.groupby(["dataset_name", "model"])
        .agg({"best/elapsed_time": ["mean", "std"], "fit_model_return_elapsed_time": ["mean", "std"]})
        .rename(columns={"best/elapsed_time": "Best Time", "fit_model_return_elapsed_time": "HPO Time"})
    )

    # Flatten multiindex columns
    df_times.columns = [" ".join(col).strip() for col in df_times.columns.values]
    # Set the same index structure as df_metrics
    df_times.index.names = ["Dataset", "Model"]

    df_times["Best Time"] = (
        df_times["Best Time mean"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
        + " $\\pm$ "
        + df_times["Best Time std"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
    )
    df_times["HPO Time"] = (
        df_times["HPO Time mean"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
        + " $\\pm$ "
        + df_times["HPO Time std"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
    )

    # Join with the existing df_metrics (verify we have the same number of rows!)
    df_metrics = df_metrics.join(df_times, how="outer")
    return df_metrics

In [ ]:
def print_one_table_per_dataset(df_metrics, hpo_metrics_rename, model_groups, datasets=None):
    df_latex = df_metrics.copy()
    df_latex = df_latex.rename(
        columns={"Best Time": "Time (s)", "Best Time mean": "Time (s) mean", "Best Time std": "Time (s) std"}
    )
    df_latex = df_latex.reset_index()
    if datasets is not None:
        df_latex = df_latex.loc[df_latex["Dataset"].isin(datasets)]
    # reapply model groups
    df_latex["Base Model"] = df_latex["Model"].apply(
        lambda x: next((group for group, models in model_groups.items() if x in models), "Other")
    )
    # redefine index with model_group
    df_latex = df_latex.set_index(["Dataset", "Base Model", "Model"])
    # sort by dataset, model_group, model
    df_latex = df_latex.sort_index(level=["Dataset", "Base Model", "Model"])

    # print per dataset
    for dataset in df_latex.index.get_level_values("Dataset").unique():
        df_print = df_latex.copy()
        df_print = df_print.loc[dataset]
        hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1]
        columns_to_hide = [
            col for col in df_latex.columns if col not in (hpo_metrics_rename + ["Time (s)"])
        ]
        columns_to_hide += hpo_metrics_to_hide
        df_print = df_print.style.hide(columns_to_hide, axis=1)
        for col in hpo_metrics_rename + ["Time (s)"]:
            highlight_metric = partial(highlight_max, column_name=f"{col} mean")
            underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
            if col in ["Davies-Bouldin", "Time (s)"]:
                highlight_metric = partial(highlight_min, column_name=f"{col} mean")
                underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
            (
                df_print.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None).apply(
                    underline_2nd_metric, subset=[col, f"{col} mean"], axis=None
                )
            )

        latex_output = df_print.to_latex(
            hrules=True,
            clines="skip-last;data",
            convert_css=True,
            column_format="ll" + "l" * (len(df_print.columns) - len(columns_to_hide)),
            # environment="longtable",
            caption=f"Clustering results on dataset {dataset}",
        )

        # fix header
        columns = df_print.index.names + [col for col in df_print.columns if col not in columns_to_hide]
        header_line = " & ".join(columns) + r" \\"

        # split into lines
        latex_output = latex_output.splitlines()
        # remove 5th and 6th line and replace with header_line
        latex_output = latex_output[:4] + [header_line] + latex_output[6:]
        # remove last cline
        latex_output = latex_output[:-4] + latex_output[-3:]

        latex_output = "\n".join(latex_output)

        print(latex_output)
        print("\n\n")

In [ ]:
def print_single_table(df_metrics, hpo_metrics_rename, datasets=None):
    # def print_single_table(df_metrics, hpo_metrics_rename, datasets=None):
    df_latex = df_metrics.copy()
    df_latex = df_latex.rename(
        columns={"Best Time": "Time (s)", "Best Time mean": "Time (s) mean", "Best Time std": "Time (s) std"}
    )
    df_latex = df_latex.reset_index()
    if datasets is not None:
        df_latex = df_latex.loc[df_latex["Dataset"].isin(datasets)]
    df_latex = df_latex.set_index(["Dataset", "Model"])
    hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1]
    columns_to_hide = [col for col in df_latex.columns if col not in (hpo_metrics_rename + ["Time (s)"])]
    columns_to_hide += hpo_metrics_to_hide
    df_latex = df_latex.style.hide(columns_to_hide, axis=1)
    for col in hpo_metrics_rename + ["Time (s)"]:
        highlight_metric = partial(highlight_max, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
        if col in ["Davies-Bouldin", "Time (s)"]:
            highlight_metric = partial(highlight_min, column_name=f"{col} mean")
            underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
        (
            df_latex.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None).apply(
                underline_2nd_metric, subset=[col, f"{col} mean"], axis=None
            )
        )

    environment = "longtable"
    latex_output = df_latex.to_latex(
        hrules=True,
        clines="skip-last;data",
        convert_css=True,
        column_format="ll" + "l" * (len(df_latex.columns) - len(columns_to_hide)),
        environment=environment,
    )

    # fix header
    columns = df_latex.index.names + [col for col in df_latex.columns if col not in columns_to_hide]
    header_line = " & ".join(columns) + r" \\"
    latex_output = latex_output.splitlines()
    if environment is None:
        # remove 3th and 4th line and replace with header_line
        latex_output = latex_output[:2] + [header_line] + latex_output[4:]
    else:
        # remove 3rd and 4th line and 8th and 9th line and replace with header_line
        latex_output = latex_output[:2] + [header_line] + latex_output[4:7] + [header_line] + latex_output[9:]

    latex_output = "\n".join(latex_output)
    print(latex_output)

# Composite per dataset and model family

In [ ]:
model_names = {
    "BatchCoHiRF-1iter-random-60": "BatchCoHiRF-sg",
    "BatchCoHiRF-1iter-random-top-down-60": "R-BatchCoHiRF-sg",
    "BatchCoHiRF-DBSCAN-1iter-random-60": "BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-DBSCAN-1iter-random-top-down-60": "R-BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-KernelRBF-1iter-random-60": "BatchCoHiRF-KernelRBF-sg",
    "BatchCoHiRF-KernelRBF-1iter-random-top-down-60": "R-BatchCoHiRF-KernelRBF-sg",
    "BatchCoHiRF-SC-SRGF-1R-1iter-random-60": "BatchCoHiRF-SC-SRGF-1R-sg",
    "BatchCoHiRF-SC-SRGF-2R-1iter-random-60": "BatchCoHiRF-SC-SRGF-2R-sg",
    "BatchCoHiRF-SC-SRGF-top-down-1iter-random-60": "R-BatchCoHiRF-SC-SRGF-sg",
	"BatchCoHiRF-SC-SRGF-1iter-random-60": "BatchCoHiRF-SC-SRGF-sg",
    "CoHiRF-60": "CoHiRF-sg",
    "CoHiRF-top-down-60": "R-CoHiRF-sg",
    "CoHiRF-DBSCAN-60": "CoHiRF-DBSCAN",
    "CoHiRF-DBSCAN-top-down-60": "R-CoHiRF-DBSCAN",
    "CoHiRF-KernelRBF-60": "CoHiRF-KernelRBF-sg",
    "CoHiRF-KernelRBF-top-down-60": "R-CoHiRF-KernelRBF-sg",
    "CoHiRF-SC-SRGF-1R-60": "CoHiRF-SC-SRGF-1R-sg",
    "CoHiRF-SC-SRGF-2R-60": "CoHiRF-SC-SRGF-2R-sg",
    "CoHiRF-SC-SRGF-60": "CoHiRF-SC-SRGF-sg",
    "CoHiRF-SC-SRGF-top-down-60": "R-CoHiRF-SC-SRGF-sg",
    "DBSCAN-60": "DBSCAN",
    "KMeans-60": "KMeans-lg",
    "KernelRBFKMeans-60": "KernelRBFKMeans-lg",
    "SpectralSubspaceRandomization-60": "SC-SRGF-lg",
    "CoHiRF-1R-60": "CoHiRF-1R-sg",
    "CoHiRF-2R-60": "CoHiRF-2R-sg",
    "CoHiRF-DBSCAN-1R-60": "CoHiRF-DBSCAN-1R",
    "CoHiRF-DBSCAN-2R-60": "CoHiRF-DBSCAN-2R",
    "CoHiRF-KernelRBF-1R-60": "CoHiRF-KernelRBF-1R-sg",
    "CoHiRF-KernelRBF-2R-60": "CoHiRF-KernelRBF-2R-sg",
    "CoHiRF-SC-SRGF-1R-60": "CoHiRF-SC-SRGF-1R-sg",
    "CoHiRF-SC-SRGF-2R-60": "CoHiRF-SC-SRGF-2R-sg",
    "BatchCoHiRF-DBSCAN-1R-1iter-random-60": "BatchCoHiRF-DBSCAN-1R",
    "BatchCoHiRF-DBSCAN-2R-1iter-random-60": "BatchCoHiRF-DBSCAN-2R",
    # 2
    "CoHiRF-SC-SRGF2-1R-60": "CoHiRF-SC-SRGF-1R-lg",
    "CoHiRF-SC-SRGF2-2R-60": "CoHiRF-SC-SRGF-2R-lg",
    "CoHiRF-SC-SRGF2-60": "CoHiRF-SC-SRGF-lg",
    "CoHiRF-SC-SRGF2-top-down-60": "R-CoHiRF-SC-SRGF-lg",
    "CoHiRF2-1R-60": "CoHiRF-1R-lg",
    "CoHiRF2-2R-60": "CoHiRF-2R-lg",
    "CoHiRF2-60": "CoHiRF-lg",
    "CoHiRF2-top-down-60": "R-CoHiRF-lg",
    "CoHiRF-KernelRBF2-1R-60": "CoHiRF-KernelRBF-1R-lg",
    "CoHiRF-KernelRBF2-2R-60": "CoHiRF-KernelRBF-2R-lg",
    "CoHiRF-KernelRBF2-60": "CoHiRF-KernelRBF-lg",
    "CoHiRF-KernelRBF2-top-down-60": "R-CoHiRF-KernelRBF-lg",
    "BatchCoHiRF-SC-SRGF2-1R-1iter-random-60": "BatchCoHiRF-SC-SRGF-1R-lg",
    "BatchCoHiRF-SC-SRGF2-2R-1iter-random-60": "BatchCoHiRF-SC-SRGF-2R-lg",
    "BatchCoHiRF-SC-SRGF2-1iter-random-60": "BatchCoHiRF-SC-SRGF-lg",
    "BatchCoHiRF-SC-SRGF2-top-down-1iter-random-60": "R-BatchCoHiRF-SC-SRGF-lg",
    "KMeans2-60": "KMeans-sg",
    "KernelRBFKMeans2-60": "KernelRBFKMeans-sg",
    "SpectralSubspaceRandomization2-60": "SC-SRGF-sg",
}


dataset_id = [
    46773,
    46779,
    1568,
    40685,
    46783,
]

hpo_metrics = [
	"adjusted_rand",
    "calinski_harabasz_score",
	"silhouette",
]

# Filter to only standardized runs
df = df_runs_parents.copy()
df = df.loc[df['standardize'] == True]
df = df.loc[df['model'].isin(model_names.keys())]
df = df.loc[df["dataset_id"].isin(dataset_id)]
df = df.loc[df['hpo_metric'].isin(hpo_metrics)]

# filter data from experiment with n_samples_representatives
n_samples_representative = [100, 1000, 5000, 10000, 20000]
ds_id = 40685
df = df.loc[~((df['dataset_id'] == ds_id) & (df["model_params/n_samples_representative"].isin(n_samples_representative)))]


df = df.replace({"model": model_names})
df = df.replace({"dataset_name": dataset_names})

# Filter to only runs with hpo_seed in range(5)
df = df.loc[df['hpo_seed'].isin(range(5))]

# Filter to only show batch methods for datasets with more than 1000 instances
df = df.loc[~((df['n_instances'] < 1000) & (df['model'].str.find('Batch') != -1))]

# define group of models
model_groups = {
    "KMeans": [
        "KMeans-sg",
        "CoHiRF-sg",
        "R-CoHiRF-sg",
        "BatchCoHiRF-sg",
        "R-BatchCoHiRF-sg",
        "CoHiRF-1R-sg",
        "CoHiRF-2R-sg",
        "KMeans-lg",
        "CoHiRF-lg",
        "R-CoHiRF-lg",
        "BatchCoHiRF-lg",
        "R-BatchCoHiRF-lg",
        "CoHiRF-1R-lg",
        "CoHiRF-2R-lg",
    ],
    "KernelKMeans": [
        "KernelRBFKMeans-sg",
        "CoHiRF-KernelRBF-sg",
        "R-CoHiRF-KernelRBF-sg",
        "BatchCoHiRF-KernelRBF-sg",
        "R-BatchCoHiRF-KernelRBF-sg",
        "CoHiRF-KernelRBF-1R-sg",
        "CoHiRF-KernelRBF-2R-sg",
        "KernelRBFKMeans-lg",
        "CoHiRF-KernelRBF-lg",
        "R-CoHiRF-KernelRBF-lg",
        "BatchCoHiRF-KernelRBF-lg",
        "R-BatchCoHiRF-KernelRBF-lg",
        "CoHiRF-KernelRBF-1R-lg",
        "CoHiRF-KernelRBF-2R-lg",
    ],
    "DBSCAN": [
        "DBSCAN",
        "CoHiRF-DBSCAN",
        "R-CoHiRF-DBSCAN",
        "BatchCoHiRF-DBSCAN",
        "R-BatchCoHiRF-DBSCAN",
        "CoHiRF-DBSCAN-1R",
        "CoHiRF-DBSCAN-2R",
        "BatchCoHiRF-DBSCAN-1R",
        "BatchCoHiRF-DBSCAN-2R",
    ],
    "SC-SRGF": [
        "SC-SRGF-sg",
        "CoHiRF-SC-SRGF-sg",
        "R-CoHiRF-SC-SRGF-sg",
        "CoHiRF-SC-SRGF-1R-sg",
        "CoHiRF-SC-SRGF-2R-sg",
        "BatchCoHiRF-SC-SRGF-sg",
        "R-BatchCoHiRF-SC-SRGF-sg",
        "BatchCoHiRF-SC-SRGF-1R-sg",
        "BatchCoHiRF-SC-SRGF-2R-sg",
        "SC-SRGF-lg",
        "CoHiRF-SC-SRGF-lg",
        "R-CoHiRF-SC-SRGF-lg",
        "CoHiRF-SC-SRGF-1R-lg",
        "CoHiRF-SC-SRGF-2R-lg",
        "BatchCoHiRF-SC-SRGF-lg",
        "R-BatchCoHiRF-SC-SRGF-lg",
        "BatchCoHiRF-SC-SRGF-1R-lg",
        "BatchCoHiRF-SC-SRGF-2R-lg",
    ],
}
df['model_group'] = df['model'].apply(lambda x: next((group for group, models in model_groups.items() if x in models), 'Other'))

# re-scale some metrics and build composite metric
# re-scale ari to be between 0 and 1 (originally between -0.5 and 1), by considering everything below 0 as 0
df["best/adjusted_rand_rescaled"] = df["best/adjusted_rand"].apply(lambda x: 0.0 if x < 0 else x)

# re-scale silhouette to be between 0 and 1 (originally between -1 and 1)
df["best/silhouette_rescaled"] = (df["best/silhouette"] - (-1)) / (1 - (-1)) 

# re-scale calinski to be between 0 and 1 normalized by dataset, model_group and hpo_metric
# replace calinksi -1.0 with 0.0
df["best/calinski_harabasz_score"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
# df["best/calinski_harabasz_score_rescaled"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
df["best/calinski_harabasz_score_rescaled"] = df.groupby(["dataset_id", "model_group", "hpo_metric"])[
    "best/calinski_harabasz_score"
].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else (0.0 if x.max() == 0 else 1.0))

hpo_metrics = [
    "adjusted_rand",
    "adjusted_rand_rescaled",
    "calinski_harabasz_score",
    "calinski_harabasz_score_rescaled",
    "silhouette",
    "silhouette_rescaled",
]

hpo_metrics_rename = [
    "ARI",
    "Rescaled ARI",
    "Calinski",
    "Rescaled Calinski",
    "Silhouette",
    "Rescaled Silhouette",
]
df_metrics = get_df_metrics(df, hpo_metrics, hpo_metrics_rename)

# One table per dataset

In [ ]:
print_one_table_per_dataset(df_metrics, hpo_metrics_rename, model_groups)

In [ ]:
datasets = [
    "alizadeh-2000-v2",
    "garber-2001",
    "bittner-2000",
    "nursery",
    "shuttle",
    "coil-20",
    "chowdary-2006",
]
print_one_table_per_dataset(df_metrics, hpo_metrics_rename, model_groups, datasets=datasets)

# ARI AND TIME ONLY

In [ ]:
datasets = [
    "alizadeh-2000-v2",
    "garber-2001",
    "bittner-2000",
    "nursery",
    "shuttle",
    "mnist",
    "coil-20",
    "chowdary-2006",
]
hpo_metrics_rename2 = [
    "ARI",
]
print_one_table_per_dataset(df_metrics, hpo_metrics_rename2, model_groups, datasets=datasets)

# Composite per dataset

In [ ]:
model_names = {
    "BatchCoHiRF-1iter-random-60": "BatchCoHiRF",
    "BatchCoHiRF-1iter-random-top-down-60": "R-BatchCoHiRF",
    "BatchCoHiRF-DBSCAN-1iter-random-60": "BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-DBSCAN-1iter-random-top-down-60": "R-BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-KernelRBF-1iter-random-60": "BatchCoHiRF-KernelRBF",
    "BatchCoHiRF-KernelRBF-1iter-random-top-down-60": "R-BatchCoHiRF-KernelRBF",
    "BatchCoHiRF-SC-SRGF-1R-1iter-random-60": "BatchCoHiRF-SC-SRGF",
    "CoHiRF-60": "CoHiRF",
    "CoHiRF-top-down-60": "R-CoHiRF",
    "CoHiRF-DBSCAN-60": "CoHiRF-DBSCAN",
    "CoHiRF-DBSCAN-top-down-60": "R-CoHiRF-DBSCAN",
    "CoHiRF-KernelRBF-60": "CoHiRF-KernelRBF",
    "CoHiRF-KernelRBF-top-down-60": "R-CoHiRF-KernelRBF",
    "CoHiRF-SC-SRGF-1R-60": "CoHiRF-SC-SRGF-1R",
    "CoHiRF-SC-SRGF-2R-60": "CoHiRF-SC-SRGF-2R",
    "DBSCAN-60": "DBSCAN",
    "KMeans-60": "KMeans",
    "KernelRBFKMeans-60": "KernelRBFKMeans",
    "SpectralSubspaceRandomization-60": "SC-SRGF",
}

dataset_names = {
    "binary_alpha_digits": "binary-alpha-digits",
    "mnist_784": "mnist",
}  # otherwise we get an error in latex

dataset_id = [
    61,
    46773,
    46776,
    46778,
    46779,
    46782,
    46783,
    554,
    40685,
    1568,
    47039,
]

hpo_metrics = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]

# Filter to only standardized runs
df = df_runs_parents.copy()
df = df.loc[df["standardize"] == True]
df = df.loc[df["model"].isin(model_names.keys())]
df = df.loc[df["dataset_id"].isin(dataset_id)]
df = df.loc[df["hpo_metric"].isin(hpo_metrics)]
df = df.replace({"model": model_names})
df = df.replace({"dataset_name": dataset_names})

# Filter to only runs with hpo_seed in range(5)
df = df.loc[df["hpo_seed"].isin(range(5))]

# Filter to only show batch methods for datasets with more than 1000 instances
df = df.loc[~((df["n_instances"] < 1000) & (df["model"].str.find("Batch") != -1))]

# define group of models
model_groups = {
    "KMeans": [
        "KMeans",
        "CoHiRF",
        "R-CoHiRF",
        "CoHiRF-1000",
        "BatchCoHiRF",
        "R-BatchCoHiRF",
        "BatchCoHiRF-nolaststop",
        "R-BatchCoHiRF-nolaststop",
    ],
    "KernelKMeans": [
        "KernelRBFKMeans",
        "CoHiRF-KernelRBF",
        "R-CoHiRF-KernelRBF",
        "BatchCoHiRF-KernelRBF",
        "R-BatchCoHiRF-KernelRBF",
        "BatchCoHiRF-KernelRBF-nolaststop",
        "R-BatchCoHiRF-KernelRBF-nolaststop",
    ],
    "DBSCAN": [
        "DBSCAN",
        "CoHiRF-DBSCAN",
        "R-CoHiRF-DBSCAN",
        "BatchCoHiRF-DBSCAN",
        "R-BatchCoHiRF-DBSCAN",
        "BatchCoHiRF-DBSCAN-nolaststop",
        "R-BatchCoHiRF-DBSCAN-nolaststop",
    ],
    "SC-SRGF": [
        "SC-SRGF",
        "CoHiRF-SC-SRGF",
        "CoHiRF-SC-SRGF-1R",
        "CoHiRF-SC-SRGF-2R",
        "BatchCoHiRF-SC-SRGF",
        "BatchCoHiRF-SC-SRGF-2R",
        "BatchCoHiRF-SC-SRGF-nolaststop",
        "BatchCoHiRF-SC-SRGF-2R-nolaststop",
    ],
}
df["model_group"] = df["model"].apply(
    lambda x: next((group for group, models in model_groups.items() if x in models), "Other")
)

# re-scale some metrics and build composite metric
# re-scale ari to be between 0 and 1 (originally between -0.5 and 1), by considering everything below 0 as 0
df["best/adjusted_rand_rescaled"] = df["best/adjusted_rand"].apply(lambda x: 0.0 if x < 0 else x)

# re-scale silhouette to be between 0 and 1 (originally between -1 and 1)
df["best/silhouette_rescaled"] = (df["best/silhouette"] - (-1)) / (1 - (-1))

# re-scale calinski to be between 0 and 1 normalized by dataset, model_group and hpo_metric
# replace calinksi -1.0 with 0.0
df["best/calinski_harabasz_score"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
# df["best/calinski_harabasz_score_rescaled"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
df["best/calinski_harabasz_score_rescaled"] = df.groupby(["dataset_id", "hpo_metric"])[
    "best/calinski_harabasz_score"
].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else (0.0 if x.max() == 0 else 1.0))

hpo_metrics = [
    "adjusted_rand",
    "adjusted_rand_rescaled",
    "calinski_harabasz_score",
    "calinski_harabasz_score_rescaled",
    "silhouette",
    "silhouette_rescaled",
]

hpo_metrics_rename = [
    "ARI",
    "Rescaled ARI",
    "Calinski",
    "Rescaled Calinski",
    "Silhouette",
    "Rescaled Silhouette",
]
df_metrics = get_df_metrics(df, hpo_metrics, hpo_metrics_rename)

In [ ]:
print_one_table_per_dataset(df_metrics, hpo_metrics_rename, model_groups, datasets=None)

In [ ]:
datasets = [
    "alizadeh-2000-v2",
    "garber-2001",
    "bittner-2000",
    "nursery",
    "shuttle",
    "mnist",
    "coil-20",
    "chowdary-2006",
]
print_one_table_per_dataset(df_metrics, hpo_metrics_rename, model_groups, datasets=datasets)

# Single table

In [ ]:
print_single_table(df_metrics, hpo_metrics_rename, datasets=None)

In [ ]:
datasets = [
    "alizadeh-2000-v2",
    "garber-2001",
    "bittner-2000",
    "nursery",
    "shuttle",
    "mnist",
    "coil-20",
    "chowdary-2006",
]
print_single_table(df_metrics, hpo_metrics_rename, datasets=datasets)

# per dataset per model group

In [ ]:
df_latex = df_metrics.copy()
df_latex = df_latex.reset_index()
# reapply model groups
df_latex["Base Model"] = df_latex["Model"].apply(
    lambda x: next((group for group, models in model_groups.items() if x in models), "Other")
)
# redefine index with model_group
df_latex = df_latex.set_index(["Dataset", "Base Model", "Model"])
# sort by dataset, model_group, model
df_latex = df_latex.sort_index(level=["Dataset", "Base Model", "Model"])


# print per dataset
for dataset in df_latex.index.get_level_values("Dataset").unique():
    df_print = df_latex.copy()
    df_print = df_print.loc[dataset]
    hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1]
    columns_to_hide = [col for col in df_latex.columns if col not in (hpo_metrics_rename + ["Best Time", "HPO Time"])]
    columns_to_hide += hpo_metrics_to_hide
    df_print = df_print.style.hide(columns_to_hide, axis=1)
    for col in hpo_metrics_rename + ["Best Time", "HPO Time"]:
        highlight_metric = partial(highlight_max, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
        if col in ["Davies-Bouldin", "Best Time", "HPO Time"]:
            highlight_metric = partial(highlight_min, column_name=f"{col} mean")
            underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
        (
            df_print.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None).apply(
                underline_2nd_metric, subset=[col, f"{col} mean"], axis=None
            )
        )

    latex_output = df_print.to_latex(
        hrules=True,
        clines="skip-last;data",
        convert_css=True,
        column_format="ll" + "l" * (len(df_print.columns) - len(columns_to_hide)),
        # environment="longtable",
        caption=f"Clustering results on dataset {dataset}",
    )

    # fix header
    columns = df_print.index.names + [col for col in df_print.columns if col not in columns_to_hide]
    header_line = " & ".join(columns) + r" \\"

    # split into lines
    latex_output = latex_output.splitlines()
    # remove 5th and 6th line and replace with header_line
    latex_output = latex_output[:4] + [header_line] + latex_output[6:]
    # remove last cline
    latex_output = latex_output[:-4] + latex_output[-3:]

    latex_output = "\n".join(latex_output)

    print(latex_output)
    print("\n\n")

In [ ]:
df_latex = df_metrics.copy()
df_latex = df_latex.reset_index()
datasets = ["alizadeh-2000-v2", "garber-2001", "bittner-2000", "nursery", "shuttle", "mnist", "coil-20", "chowdary-2006"]
df_latex = df_latex.loc[df_latex["Dataset"].isin(datasets)]
# reapply model groups
df_latex["Base Model"] = df_latex["Model"].apply(
    lambda x: next((group for group, models in model_groups.items() if x in models), "Other")
)
# redefine index with model_group
df_latex = df_latex.set_index(["Dataset", "Base Model", "Model"])
# sort by dataset, model_group, model
df_latex = df_latex.sort_index(level=["Dataset", "Base Model", "Model"])


# print per dataset
for dataset in df_latex.index.get_level_values("Dataset").unique():
    df_print = df_latex.copy()
    df_print = df_print.loc[dataset]
    hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1]
    columns_to_hide = [col for col in df_latex.columns if col not in (hpo_metrics_rename + ["Best Time", "HPO Time"])]
    columns_to_hide += hpo_metrics_to_hide
    df_print = df_print.style.hide(columns_to_hide, axis=1)
    for col in hpo_metrics_rename + ["Best Time", "HPO Time"]:
        highlight_metric = partial(highlight_max, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
        if col in ["Davies-Bouldin", "Best Time", "HPO Time"]:
            highlight_metric = partial(highlight_min, column_name=f"{col} mean")
            underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
        (
            df_print.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None).apply(
                underline_2nd_metric, subset=[col, f"{col} mean"], axis=None
            )
        )

    latex_output = df_print.to_latex(
        hrules=True,
        clines="skip-last;data",
        convert_css=True,
        column_format="ll" + "l" * (len(df_print.columns) - len(columns_to_hide)),
        # environment="longtable",
        caption=f"Clustering results on dataset {dataset}",
    )

    # fix header
    columns = df_print.index.names + [col for col in df_print.columns if col not in columns_to_hide]
    header_line = " & ".join(columns) + r" \\"

    # split into lines
    latex_output = latex_output.splitlines()
    # remove 5th and 6th line and replace with header_line
    latex_output = latex_output[:4] + [header_line] + latex_output[6:]
    # remove last cline
    latex_output = latex_output[:-4] + latex_output[-3:]

    latex_output = "\n".join(latex_output)

    print(latex_output)
    print("\n\n")

In [ ]:
df_latex = df_metrics.copy()
df_latex = df_latex.reset_index()
# reapply model groups
df_latex['Base Model'] = df_latex['Model'].apply(lambda x: next((group for group, models in model_groups.items() if x in models), 'Other'))
# redefine index with model_group
df_latex = df_latex.set_index(['Dataset', 'Base Model', 'Model'])
# sort by dataset, model_group, model
df_latex = df_latex.sort_index(level=['Dataset', 'Base Model', 'Model'])


# print per dataset
for dataset in df_latex.index.get_level_values('Dataset').unique():
    df_print = df_latex.copy()
    df_print = df_print.loc[dataset]
    hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1] 
    columns_to_hide = [col for col in df_latex.columns if col not in (hpo_metrics_rename + ["Best Time", "HPO Time"])]
    columns_to_hide += hpo_metrics_to_hide
    df_print = df_print.style.hide(columns_to_hide, axis=1)
    for col in hpo_metrics_rename + ["Best Time", "HPO Time"]:
        highlight_metric = partial(highlight_max, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
        if col in ["Davies-Bouldin", "Best Time", "HPO Time"]:
            highlight_metric = partial(highlight_min, column_name=f"{col} mean")
            underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
        (
            df_print.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None).apply(
                underline_2nd_metric, subset=[col, f"{col} mean"], axis=None
            )
        )

    df_print = df_print.hide(level=0, axis=0)
    latex_output = (
        df_print.to_latex(
            hrules=True,
            clines="skip-last;data",
            convert_css=True,
            column_format="ll" + "l" * (len(df_print.columns) - len(columns_to_hide)),
            # environment="longtable",
            caption=f"Clustering results on dataset {dataset}",
        )
    )

    # fix header
    columns = df_print.index.names[:1] + [col for col in df_print.columns if col not in columns_to_hide]
    header_line = ' & '.join(columns) + r' \\'

    # split into lines
    latex_output = latex_output.splitlines()
    # remove 5th and 6th line and replace with header_line
    latex_output = latex_output[:4] + [header_line] + latex_output[6:]
    latex_output = '\n'.join(latex_output)

    # manually add clines after model groups
    model_groups_in_data = df_print.index.get_level_values('Base Model').unique().tolist()
    lines = latex_output.splitlines()
    new_lines = []
    last_line = ""
    for i, line in enumerate(lines[6:-3]): # skip first 6 lines and last 3 lines
        model_last_line = last_line.split('&')[0].strip()
        model_current_line = line.split('&')[0].strip()
        model_group_last_line = next((group for group, models in model_groups.items() if model_last_line in models), 'Other')
        model_group_current_line = next((group for group, models in model_groups.items() if model_current_line in models), 'Other')
        if model_group_last_line != model_group_current_line and i != 0:
            new_lines.append(r'\cline{' + f'1-{len(columns)}' + r'}')
        new_lines.append(line)
        last_line = line

    latex_output = '\n'.join(lines[:6] + new_lines + lines[-3:])

    print(latex_output)
    print("\n\n")
    print("\pagebreak")

# Composite per dataset

In [ ]:
model_names = {
    "BatchCoHiRF-1iter-random-60": "BatchCoHiRF",
    "BatchCoHiRF-1iter-random-top-down-60": "R-BatchCoHiRF",
    "BatchCoHiRF-DBSCAN-1iter-random-60": "BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-DBSCAN-1iter-random-top-down-60": "R-BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-KernelRBF-1iter-random-60": "BatchCoHiRF-KernelRBF",
    "BatchCoHiRF-KernelRBF-1iter-random-top-down-60": "R-BatchCoHiRF-KernelRBF",
    "BatchCoHiRF-SC-SRGF-1R-1iter-random-60": "BatchCoHiRF-SC-SRGF",
    "CoHiRF-60": "CoHiRF",
    "CoHiRF-top-down-60": "R-CoHiRF",
    "CoHiRF-DBSCAN-60": "CoHiRF-DBSCAN",
    "CoHiRF-DBSCAN-top-down-60": "R-CoHiRF-DBSCAN",
    "CoHiRF-KernelRBF-60": "CoHiRF-KernelRBF",
    "CoHiRF-KernelRBF-top-down-60": "R-CoHiRF-KernelRBF",
    "CoHiRF-SC-SRGF-1R-60": "CoHiRF-SC-SRGF-1R",
    "CoHiRF-SC-SRGF-2R-60": "CoHiRF-SC-SRGF-2R",
    "DBSCAN-60": "DBSCAN",
    "KMeans-60": "KMeans",
    "KernelRBFKMeans-60": "KernelRBFKMeans",
    "SpectralSubspaceRandomization-60": "SC-SRGF",
}

dataset_names = {
    "binary_alpha_digits": "binary-alpha-digits",
	"mnist_784": "mnist",
}  # otherwise we get an error in latex

dataset_id = [
    61,
    46773,
    46776,
    46778,
    46779,
    46782,
    46783,
    554,
    40685,
    1568,
	47039,
]

hpo_metrics = [
	"adjusted_rand",
    "calinski_harabasz_score",
	"silhouette",
]

# Filter to only standardized runs
df = df_runs_parents.copy()
df = df.loc[df['standardize'] == True]
df = df.loc[df['model'].isin(model_names.keys())]
df = df.loc[df["dataset_id"].isin(dataset_id)]
df = df.loc[df['hpo_metric'].isin(hpo_metrics)]
df = df.replace({"model": model_names})
df = df.replace({"dataset_name": dataset_names})

# Filter to only runs with hpo_seed in range(5)
df = df.loc[df['hpo_seed'].isin(range(5))]

# Filter to only show batch methods for datasets with more than 1000 instances
df = df.loc[~((df['n_instances'] < 1000) & (df['model'].str.find('Batch') != -1))]

# define group of models
model_groups = {
	"KMeans": ["KMeans", "CoHiRF", "R-CoHiRF", "CoHiRF-1000", "BatchCoHiRF", "R-BatchCoHiRF", "BatchCoHiRF-nolaststop", "R-BatchCoHiRF-nolaststop"],
	"KernelKMeans": ["KernelRBFKMeans", "CoHiRF-KernelRBF", "R-CoHiRF-KernelRBF", "BatchCoHiRF-KernelRBF", "R-BatchCoHiRF-KernelRBF", "BatchCoHiRF-KernelRBF-nolaststop", "R-BatchCoHiRF-KernelRBF-nolaststop"],
    "DBSCAN": ["DBSCAN", "CoHiRF-DBSCAN", "R-CoHiRF-DBSCAN", "BatchCoHiRF-DBSCAN", "R-BatchCoHiRF-DBSCAN", "BatchCoHiRF-DBSCAN-nolaststop", "R-BatchCoHiRF-DBSCAN-nolaststop"],
    "SC-SRGF": ["SC-SRGF", "CoHiRF-SC-SRGF", "CoHiRF-SC-SRGF-1R", "CoHiRF-SC-SRGF-2R", "BatchCoHiRF-SC-SRGF", "BatchCoHiRF-SC-SRGF-2R", "BatchCoHiRF-SC-SRGF-nolaststop", "BatchCoHiRF-SC-SRGF-2R-nolaststop"],
}
df['model_group'] = df['model'].apply(lambda x: next((group for group, models in model_groups.items() if x in models), 'Other'))


# re-scale some metrics and build composite metric
# re-scale ari to be between 0 and 1 (originally between -0.5 and 1), by considering everything below 0 as 0
df["best/adjusted_rand_rescaled"] = df["best/adjusted_rand"].apply(lambda x: 0.0 if x < 0 else x)

# re-scale silhouette to be between 0 and 1 (originally between -1 and 1)
df["best/silhouette_rescaled"] = (df["best/silhouette"] - (-1)) / (1 - (-1)) 

# re-scale calinski to be between 0 and 1 normalized by dataset, and hpo_metric
# replace calinksi -1.0 with 0.0
df["best/calinski_harabasz_score"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
# df["best/calinski_harabasz_score_rescaled"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
df["best/calinski_harabasz_score_rescaled"] = df.groupby(["dataset_id", "hpo_metric"])[
    "best/calinski_harabasz_score"
].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else (0.0 if x.max() == 0 else 1.0))

In [ ]:
hpo_metrics = [
    "adjusted_rand",
    "adjusted_rand_rescaled",
    "calinski_harabasz_score",
    "calinski_harabasz_score_rescaled",
    "silhouette",
    "silhouette_rescaled",
]

hpo_metrics_rename = [
    "ARI",
    "Rescaled ARI",
    "Calinski",
    "Rescaled Calinski",
    "Silhouette",
    "Rescaled Silhouette",
]

dfs_metrics = {}

for hpo_metric, hpo_metric_rename in zip(hpo_metrics, hpo_metrics_rename):
    if hpo_metric.find("_rescaled") != -1:
        original_metric = hpo_metric.replace("_rescaled", "")
    else:
        original_metric = hpo_metric
    df_metric = df.loc[df["hpo_metric"] == original_metric][
        ["dataset_name", "model", "hpo_seed", f"best/{hpo_metric}"]
    ].rename(columns={f"best/{hpo_metric}": hpo_metric_rename})
    df_metric = df_metric.dropna(subset=[hpo_metric_rename])
    df_metric = df_metric.set_index(['dataset_name', 'model', 'hpo_seed'])
    df_metric = df_metric.astype({hpo_metric_rename: float})
    dfs_metrics[hpo_metric_rename] = df_metric

df_metrics = pd.concat(dfs_metrics.values(), axis=1, join="outer")
df_metrics = df_metrics.reset_index()

# calculate mean and std
df_metrics = df_metrics.groupby(['dataset_name', 'model']).agg(['mean', 'std'])
# flatten multiindex columns
df_metrics.columns = [' '.join(col).strip() for col in df_metrics.columns.values]
# drop hpo_seed level
df_metrics = df_metrics.drop(columns=['hpo_seed mean', 'hpo_seed std'])
# Rename index levels
df_metrics.index.names = ["Dataset", "Model"]

# create a composite metric as the average of the metrics
df_metrics["Composite Metric mean"] = df_metrics[[f"{metric} mean" for metric in hpo_metrics_rename if "Rescaled" in metric]].mean(axis=1)
df_metrics["Composite Metric std"] = 1/len(hpo_metrics_rename) * (df_metrics[[f"{metric} std" for metric in hpo_metrics_rename if "Rescaled" in metric]]**2).sum(axis=1)**0.5
hpo_metrics_rename.append("Composite Metric")


for metric in hpo_metrics_rename:
    df_metrics[f"{metric}"] = (
        df_metrics[f"{metric} mean"].apply(lambda x: f"{x:.3f}" if not pd.isna(x) else "No Run")
        + " $\\pm$ "
        + df_metrics[f"{metric} std"].apply(lambda x: f"{x:.3f}" if not pd.isna(x) else "No Run")
    )

In [ ]:
# Calculate mean and std times for each dataset-model combination across all metrics
df_times = (
    df.groupby(["dataset_name", "model"])
    .agg({"best/elapsed_time": ["mean", "std"], "fit_model_return_elapsed_time": ["mean", "std"]})
    .rename(columns={"best/elapsed_time": "Best Time", "fit_model_return_elapsed_time": "HPO Time"})
)

# Flatten multiindex columns
df_times.columns = [' '.join(col).strip() for col in df_times.columns.values]
# Set the same index structure as df_metrics
df_times.index.names = ["Dataset", "Model"]

df_times["Best Time"] = (
	df_times["Best Time mean"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
	+ " $\\pm$ " 
	+ df_times["Best Time std"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
)
df_times["HPO Time"] = (
	df_times["HPO Time mean"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
	+ " $\\pm$ "
	+ df_times["HPO Time std"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
)

# Join with the existing df_metrics (verify we have the same number of rows!)
df_metrics = df_metrics.join(df_times, how="outer")

The following will provide the latex code for a clean table, we only need to make a little adjustement in the first line to delete the "key" and have only one header. For the longtable environment (full data) we need to add the "\*" at the end of lines we dont want to have a page break. We also should replace the entire begin{table} ... end{table} by begin{longtable} ... end{longtable} in the latex file, if you want to put caption and labels you should break the line after with '\\' (put both on the same line!)


In [ ]:
df_latex = df_metrics.copy()
hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1] 
columns_to_hide = [col for col in df_latex.columns if col not in (hpo_metrics_rename + ["Best Time", "HPO Time"])]
columns_to_hide += hpo_metrics_to_hide
df_latex = df_latex.style.hide(columns_to_hide, axis=1)
for col in hpo_metrics_rename + ["Best Time", "HPO Time"]:
    highlight_metric = partial(highlight_max, column_name=f"{col} mean")
    underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
    if col in ["Davies-Bouldin", "Best Time", "HPO Time"]:
        highlight_metric = partial(highlight_min, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
    (df_latex.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None)
    .apply(underline_2nd_metric, subset=[col, f"{col} mean"], axis=None))

environment = 'longtable'
latex_output = df_latex.to_latex(
    hrules=True,
    clines="skip-last;data",
    convert_css=True,
    column_format="ll" + "l" * (len(df_latex.columns) - len(columns_to_hide)),
    environment=environment,
)

# fix header
columns = df_latex.index.names + [col for col in df_latex.columns if col not in columns_to_hide]
header_line = ' & '.join(columns) + r' \\'
latex_output = latex_output.splitlines()
if environment is None:
    # remove 3th and 4th line and replace with header_line
    latex_output = latex_output[:2] + [header_line] + latex_output[4:]
else:
    # remove 3rd and 4th line and 8th and 9th line and replace with header_line
    latex_output = latex_output[:2] + [header_line] + latex_output[4:7] + [header_line] + latex_output[9:]

latex_output = "\n".join(latex_output)
print(latex_output)

# per dataset per model group

In [ ]:
df_latex = df_metrics.copy()
df_latex = df_latex.reset_index()
# reapply model groups
df_latex["Base Model"] = df_latex["Model"].apply(
    lambda x: next((group for group, models in model_groups.items() if x in models), "Other")
)
# redefine index with model_group
df_latex = df_latex.set_index(["Dataset", "Base Model", "Model"])
# sort by dataset, model_group, model
df_latex = df_latex.sort_index(level=["Dataset", "Base Model", "Model"])


# print per dataset
for dataset in df_latex.index.get_level_values("Dataset").unique():
    df_print = df_latex.copy()
    df_print = df_print.loc[dataset]
    hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1]
    columns_to_hide = [col for col in df_latex.columns if col not in (hpo_metrics_rename + ["Best Time", "HPO Time"])]
    columns_to_hide += hpo_metrics_to_hide
    df_print = df_print.style.hide(columns_to_hide, axis=1)
    for col in hpo_metrics_rename + ["Best Time", "HPO Time"]:
        highlight_metric = partial(highlight_max, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
        if col in ["Davies-Bouldin", "Best Time", "HPO Time"]:
            highlight_metric = partial(highlight_min, column_name=f"{col} mean")
            underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
        (
            df_print.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None).apply(
                underline_2nd_metric, subset=[col, f"{col} mean"], axis=None
            )
        )

    latex_output = df_print.to_latex(
        hrules=True,
        clines="skip-last;data",
        convert_css=True,
        column_format="ll" + "l" * (len(df_print.columns) - len(columns_to_hide)),
        # environment="longtable",
        caption=f"Clustering results on dataset {dataset}",
    )

    # fix header
    columns = df_print.index.names + [col for col in df_print.columns if col not in columns_to_hide]
    header_line = " & ".join(columns) + r" \\"

    # split into lines
    latex_output = latex_output.splitlines()
    # remove 5th and 6th line and replace with header_line
    latex_output = latex_output[:4] + [header_line] + latex_output[6:]
    # remove last cline
    latex_output = latex_output[:-4] + latex_output[-3:]

    latex_output = "\n".join(latex_output)

    print(latex_output)
    print("\n\n")

In [ ]:
df_latex = df_metrics.copy()
df_latex = df_latex.reset_index()
datasets = ["alizadeh-2000-v2", "garber-2001", "bittner-2000", "nursery", "shuttle", "mnist", "coil-20", "chowdary-2006"]
df_latex = df_latex.loc[df_latex["Dataset"].isin(datasets)]
# reapply model groups
df_latex["Base Model"] = df_latex["Model"].apply(
    lambda x: next((group for group, models in model_groups.items() if x in models), "Other")
)
# redefine index with model_group
df_latex = df_latex.set_index(["Dataset", "Base Model", "Model"])
# sort by dataset, model_group, model
df_latex = df_latex.sort_index(level=["Dataset", "Base Model", "Model"])


# print per dataset
for dataset in df_latex.index.get_level_values("Dataset").unique():
    df_print = df_latex.copy()
    df_print = df_print.loc[dataset]
    hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1]
    columns_to_hide = [col for col in df_latex.columns if col not in (hpo_metrics_rename + ["Best Time", "HPO Time"])]
    columns_to_hide += hpo_metrics_to_hide
    df_print = df_print.style.hide(columns_to_hide, axis=1)
    for col in hpo_metrics_rename + ["Best Time", "HPO Time"]:
        highlight_metric = partial(highlight_max, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
        if col in ["Davies-Bouldin", "Best Time", "HPO Time"]:
            highlight_metric = partial(highlight_min, column_name=f"{col} mean")
            underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
        (
            df_print.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None).apply(
                underline_2nd_metric, subset=[col, f"{col} mean"], axis=None
            )
        )

    latex_output = df_print.to_latex(
        hrules=True,
        clines="skip-last;data",
        convert_css=True,
        column_format="ll" + "l" * (len(df_print.columns) - len(columns_to_hide)),
        # environment="longtable",
        caption=f"Clustering results on dataset {dataset}",
    )

    # fix header
    columns = df_print.index.names + [col for col in df_print.columns if col not in columns_to_hide]
    header_line = " & ".join(columns) + r" \\"

    # split into lines
    latex_output = latex_output.splitlines()
    # remove 5th and 6th line and replace with header_line
    latex_output = latex_output[:4] + [header_line] + latex_output[6:]
    # remove last cline
    latex_output = latex_output[:-4] + latex_output[-3:]

    latex_output = "\n".join(latex_output)

    print(latex_output)
    print("\n\n")

In [ ]:
df_latex = df_metrics.copy()
df_latex = df_latex.reset_index()
# reapply model groups
df_latex['Base Model'] = df_latex['Model'].apply(lambda x: next((group for group, models in model_groups.items() if x in models), 'Other'))
# redefine index with model_group
df_latex = df_latex.set_index(['Dataset', 'Base Model', 'Model'])
# sort by dataset, model_group, model
df_latex = df_latex.sort_index(level=['Dataset', 'Base Model', 'Model'])


# print per dataset
for dataset in df_latex.index.get_level_values('Dataset').unique():
    df_print = df_latex.copy()
    df_print = df_print.loc[dataset]
    hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1] 
    columns_to_hide = [col for col in df_latex.columns if col not in (hpo_metrics_rename + ["Best Time", "HPO Time"])]
    columns_to_hide += hpo_metrics_to_hide
    df_print = df_print.style.hide(columns_to_hide, axis=1)
    for col in hpo_metrics_rename + ["Best Time", "HPO Time"]:
        highlight_metric = partial(highlight_max, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
        if col in ["Davies-Bouldin", "Best Time", "HPO Time"]:
            highlight_metric = partial(highlight_min, column_name=f"{col} mean")
            underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
        (
            df_print.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None).apply(
                underline_2nd_metric, subset=[col, f"{col} mean"], axis=None
            )
        )

    df_print = df_print.hide(level=0, axis=0)
    latex_output = (
        df_print.to_latex(
            hrules=True,
            clines="skip-last;data",
            convert_css=True,
            column_format="ll" + "l" * (len(df_print.columns) - len(columns_to_hide)),
            # environment="longtable",
            caption=f"Clustering results on dataset {dataset}",
        )
    )

    # fix header
    columns = df_print.index.names[:1] + [col for col in df_print.columns if col not in columns_to_hide]
    header_line = ' & '.join(columns) + r' \\'

    # split into lines
    latex_output = latex_output.splitlines()
    # remove 5th and 6th line and replace with header_line
    latex_output = latex_output[:4] + [header_line] + latex_output[6:]
    latex_output = '\n'.join(latex_output)

    # manually add clines after model groups
    model_groups_in_data = df_print.index.get_level_values('Base Model').unique().tolist()
    lines = latex_output.splitlines()
    new_lines = []
    last_line = ""
    for i, line in enumerate(lines[6:-3]): # skip first 6 lines and last 3 lines
        model_last_line = last_line.split('&')[0].strip()
        model_current_line = line.split('&')[0].strip()
        model_group_last_line = next((group for group, models in model_groups.items() if model_last_line in models), 'Other')
        model_group_current_line = next((group for group, models in model_groups.items() if model_current_line in models), 'Other')
        if model_group_last_line != model_group_current_line and i != 0:
            new_lines.append(r'\cline{' + f'1-{len(columns)}' + r'}')
        new_lines.append(line)
        last_line = line

    latex_output = '\n'.join(lines[:6] + new_lines + lines[-3:])

    print(latex_output)
    print("\n\n")
    print("\pagebreak")

# Dataset selection in one table

In [ ]:
model_names = {
    "BatchCoHiRF-1iter-random-60": "BatchCoHiRF",
    "BatchCoHiRF-1iter-random-top-down-60": "R-BatchCoHiRF",
    "BatchCoHiRF-DBSCAN-1iter-random-60": "BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-DBSCAN-1iter-random-top-down-60": "R-BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-KernelRBF-1iter-random-60": "BatchCoHiRF-KernelRBF",
    "BatchCoHiRF-KernelRBF-1iter-random-top-down-60": "R-BatchCoHiRF-KernelRBF",
    "BatchCoHiRF-SC-SRGF-1R-1iter-random-60": "BatchCoHiRF-SC-SRGF",
    "CoHiRF-1000-60": "CoHiRF-1000",
    "CoHiRF-60": "CoHiRF",
    "CoHiRF-top-down-60": "R-CoHiRF",
    "CoHiRF-DBSCAN-60": "CoHiRF-DBSCAN",
    "CoHiRF-DBSCAN-top-down-60": "R-CoHiRF-DBSCAN",
    "CoHiRF-KernelRBF-60": "CoHiRF-KernelRBF",
    "CoHiRF-KernelRBF-top-down-60": "R-CoHiRF-KernelRBF",
    "CoHiRF-SC-SRGF-1R-60": "CoHiRF-SC-SRGF",
    # "CoHiRF-SC-SRGF-2R-60": "CoHiRF-SC-SRGF-2R",
    "DBSCAN-60": "DBSCAN",
    "KMeans-60": "KMeans",
    "KernelRBFKMeans-60": "KernelRBFKMeans",
    "SpectralSubspaceRandomization-60": "SC-SRGF",
}

dataset_names = {
    "binary_alpha_digits": "binary-alpha-digits",
    "mnist_784": "mnist",
}  # otherwise we get an error in latex

dataset_id = [
    61,
    46773,
    46776,
    46778,
    46779,
    46782,
    46783,
    554,
    40685,
    1568,
    47039,
]

hpo_metrics = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]

# Filter to only standardized runs
df = df_runs_parents.copy()
df = df.loc[df["standardize"] == True]
df = df.loc[df["model"].isin(model_names.keys())]
df = df.loc[df["dataset_id"].isin(dataset_id)]
df = df.loc[df["hpo_metric"].isin(hpo_metrics)]
df = df.replace({"model": model_names})
df = df.replace({"dataset_name": dataset_names})

# Filter to only runs with hpo_seed in range(5)
df = df.loc[df["hpo_seed"].isin(range(5))]

# Filter to only show batch methods for datasets with more than 1000 instances
df = df.loc[~((df["n_instances"] < 1000) & (df["model"].str.find("Batch") != -1))]

# define group of models
model_groups = {
    "KMeans": [
        "KMeans",
        "CoHiRF",
        "R-CoHiRF",
        "CoHiRF-1000",
        "BatchCoHiRF",
        "R-BatchCoHiRF",
        "BatchCoHiRF-nolaststop",
        "R-BatchCoHiRF-nolaststop",
    ],
    "KernelKMeans": [
        "KernelRBFKMeans",
        "CoHiRF-KernelRBF",
        "R-CoHiRF-KernelRBF",
        "BatchCoHiRF-KernelRBF",
        "R-BatchCoHiRF-KernelRBF",
        "BatchCoHiRF-KernelRBF-nolaststop",
        "R-BatchCoHiRF-KernelRBF-nolaststop",
    ],
    "DBSCAN": [
        "DBSCAN",
        "CoHiRF-DBSCAN",
        "R-CoHiRF-DBSCAN",
        "BatchCoHiRF-DBSCAN",
        "R-BatchCoHiRF-DBSCAN",
        "BatchCoHiRF-DBSCAN-nolaststop",
        "R-BatchCoHiRF-DBSCAN-nolaststop",
    ],
    "SC-SRGF": [
        "SC-SRGF",
        "CoHiRF-SC-SRGF",
        "CoHiRF-SC-SRGF-1R",
        "CoHiRF-SC-SRGF-2R",
        "BatchCoHiRF-SC-SRGF",
        "BatchCoHiRF-SC-SRGF-2R",
        "BatchCoHiRF-SC-SRGF-nolaststop",
        "BatchCoHiRF-SC-SRGF-2R-nolaststop",
    ],
}
df["model_group"] = df["model"].apply(
    lambda x: next((group for group, models in model_groups.items() if x in models), "Other")
)


# re-scale some metrics and build composite metric
# re-scale ari to be between 0 and 1 (originally between -0.5 and 1), by considering everything below 0 as 0
df["best/adjusted_rand_rescaled"] = df["best/adjusted_rand"].apply(lambda x: 0.0 if x < 0 else x)

# re-scale silhouette to be between 0 and 1 (originally between -1 and 1)
df["best/silhouette_rescaled"] = (df["best/silhouette"] - (-1)) / (1 - (-1))

# re-scale calinski to be between 0 and 1 normalized by dataset, model_group and hpo_metric
# replace calinksi -1.0 with 0.0
df["best/calinski_harabasz_score"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
# df["best/calinski_harabasz_score_rescaled"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
df["best/calinski_harabasz_score_rescaled"] = df.groupby(["dataset_id", "model_group", "hpo_metric"])[
    "best/calinski_harabasz_score"
].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else (0.0 if x.max() == 0 else 1.0))

In [ ]:
hpo_metrics = [
    "adjusted_rand",
    "adjusted_rand_rescaled",
    "calinski_harabasz_score",
    "calinski_harabasz_score_rescaled",
    "silhouette",
    "silhouette_rescaled",
]

hpo_metrics_rename = [
    "ARI",
    "Rescaled ARI",
    "Calinski",
    "Rescaled Calinski",
    "Silhouette",
    "Rescaled Silhouette",
]

dfs_metrics = {}

for hpo_metric, hpo_metric_rename in zip(hpo_metrics, hpo_metrics_rename):
    if hpo_metric.find("_rescaled") != -1:
        original_metric = hpo_metric.replace("_rescaled", "")
    else:
        original_metric = hpo_metric
    df_metric = df.loc[df["hpo_metric"] == original_metric][
        ["dataset_name", "model", "hpo_seed", f"best/{hpo_metric}"]
    ].rename(columns={f"best/{hpo_metric}": hpo_metric_rename})
    df_metric = df_metric.dropna(subset=[hpo_metric_rename])
    df_metric = df_metric.set_index(["dataset_name", "model", "hpo_seed"])
    df_metric = df_metric.astype({hpo_metric_rename: float})
    dfs_metrics[hpo_metric_rename] = df_metric

df_metrics = pd.concat(dfs_metrics.values(), axis=1, join="outer")
df_metrics = df_metrics.reset_index()

# calculate mean and std
df_metrics = df_metrics.groupby(["dataset_name", "model"]).agg(["mean", "std"])
# flatten multiindex columns
df_metrics.columns = [" ".join(col).strip() for col in df_metrics.columns.values]
# drop hpo_seed level
df_metrics = df_metrics.drop(columns=["hpo_seed mean", "hpo_seed std"])
# Rename index levels
df_metrics.index.names = ["Dataset", "Model"]

# create a composite metric as the average of the metrics
df_metrics["Composite mean"] = df_metrics[
    [f"{metric} mean" for metric in hpo_metrics_rename if "Rescaled" in metric]
].mean(axis=1)
df_metrics["Composite std"] = (
    1
    / len(hpo_metrics_rename)
    * (df_metrics[[f"{metric} std" for metric in hpo_metrics_rename if "Rescaled" in metric]] ** 2).sum(axis=1) ** 0.5
)
hpo_metrics_rename.append("Composite")


for metric in hpo_metrics_rename:
    df_metrics[f"{metric}"] = (
        df_metrics[f"{metric} mean"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "No Run")
        + " $\\pm$ "
        + df_metrics[f"{metric} std"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "No Run")
    )

In [ ]:
# Calculate mean and std times for each dataset-model combination across all metrics
df_times = (
    df.groupby(["dataset_name", "model"])
    .agg({"best/elapsed_time": ["mean", "std"], "fit_model_return_elapsed_time": ["mean", "std"]})
    .rename(columns={"best/elapsed_time": "Time (s)", "fit_model_return_elapsed_time": "HPO Time"})
)

# Flatten multiindex columns
df_times.columns = [' '.join(col).strip() for col in df_times.columns.values]
# Set the same index structure as df_metrics
df_times.index.names = ["Dataset", "Model"]

df_times["Time (s)"] = (
	df_times["Time (s) mean"].apply(lambda x: f"{x:4.2f}" if not pd.isna(x) else "No Run")
	+ " $\\pm$ " 
	+ df_times["Time (s) std"].apply(lambda x: f"{x:4.2f}" if not pd.isna(x) else "No Run")
)
df_times["HPO Time"] = (
	df_times["HPO Time mean"].apply(lambda x: f"{x:4.2f}" if not pd.isna(x) else "No Run")
	+ " $\\pm$ "
	+ df_times["HPO Time std"].apply(lambda x: f"{x:4.2f}" if not pd.isna(x) else "No Run")
)

# Join with the existing df_metrics (verify we have the same number of rows!)
df_metrics = df_metrics.join(df_times, how="outer")

In [ ]:
dataset_models = {
	"mnist": {"KMeans" : ["KMeans", "CoHiRF", "R-CoHiRF", "BatchCoHiRF", "R-BatchCoHiRF"]},
	"shuttle": {"DBSCAN" : ["DBSCAN", "CoHiRF-DBSCAN", "R-CoHiRF-DBSCAN", "BatchCoHiRF-DBSCAN", "R-BatchCoHiRF-DBSCAN"]},
	"binary-alpha-digits": {"SC-SRGF" : ["SC-SRGF", "CoHiRF-SC-SRGF"]},
	"chowdary-2006": {"SC-SRGF" : ["SC-SRGF", "CoHiRF-SC-SRGF"]},
}
df = df_metrics.copy()
df = df.reset_index()
# reapply model groups
df['Base Model'] = df['Model'].apply(lambda x: next((group for group, models in model_groups.items() if x in models), 'Other'))
df_table = []
for dataset, models_dict in dataset_models.items():
    for base_model, models in models_dict.items():
        df_subset = df.loc[(df['Dataset'] == dataset) & (df['Model'].isin(models))]
        df_table.append(df_subset)
df_table = pd.concat(df_table, axis=0)
# create column with Dataset - Model Group
df_table["Dataset / Base Model"] = (
    "\parbox{2cm}{\centering " + df_table["Dataset"] + "\\\\" + df_table["Base Model"] + "}"
)
df_table = df_table.set_index(["Dataset / Base Model", "Model"])
df_table = df_table.sort_index(level=["Dataset / Base Model", "Model"])
hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1]
columns_to_hide = [col for col in df_table.columns if col not in (hpo_metrics_rename + ["Time (s)"])]
columns_to_hide += hpo_metrics_to_hide
df_table = df_table.style.hide(columns_to_hide, axis=1)
for col in hpo_metrics_rename + ["Time (s)"]:
    highlight_metric = partial(highlight_max, column_name=f"{col} mean")
    underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
    if col in ["Davies-Bouldin", "Time (s)"]:
        highlight_metric = partial(highlight_min, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
    (df_table.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None)
    .apply(underline_2nd_metric, subset=[col, f"{col} mean"], axis=None))

latex_output = df_table.to_latex(
    hrules=True,
    clines="skip-last;data",
    convert_css=True,
    column_format="ll" + "l" * (len(df_table.columns) - len(columns_to_hide)),
    # environment="longtable",
    caption=f"Clustering results on real-world datasets.",
    # label="tab:clustering_real_world_datasets",
)

# fix header
columns = df_table.index.names + [col for col in df_table.columns if col not in columns_to_hide]
header_line = ' & '.join(columns) + r' \\'

# split into lines
latex_output = latex_output.splitlines()
# remove 5th and 6th line and replace with header_line
latex_output = latex_output[:4] + [header_line] + latex_output[6:]
# remove last cline
latex_output = latex_output[:-4] + latex_output[-3:]
# add \fontsize{5}{10}\selectfont to table
latex_output.insert(2, r'\fontsize{5}{10}\selectfont')
latex_output = "\n".join(latex_output)


print(latex_output)

# Iris with KMeans, KernelRBFKMeans

In [ ]:
model_names = {
    "BatchCoHiRF-1iter-random-60": "BatchCoHiRF",
    "BatchCoHiRF-1iter-random-top-down-60": "R-BatchCoHiRF",
    "BatchCoHiRF-DBSCAN-1iter-random-60": "BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-DBSCAN-1iter-random-top-down-60": "R-BatchCoHiRF-DBSCAN",
    "BatchCoHiRF-KernelRBF-1iter-random-60": "BatchCoHiRF-KernelRBF",
    "BatchCoHiRF-KernelRBF-1iter-random-top-down-60": "R-BatchCoHiRF-KernelRBF",
    "BatchCoHiRF-SC-SRGF-1R-1iter-random-60": "BatchCoHiRF-SC-SRGF",
    "CoHiRF-1000-60": "CoHiRF-1000",
    "CoHiRF-60": "CoHiRF",
    "CoHiRF-top-down-60": "R-CoHiRF",
    "CoHiRF-DBSCAN-60": "CoHiRF-DBSCAN",
    "CoHiRF-DBSCAN-top-down-60": "R-CoHiRF-DBSCAN",
    "CoHiRF-KernelRBF-60": "CoHiRF-KernelRBF",
    "CoHiRF-KernelRBF-top-down-60": "R-CoHiRF-KernelRBF",
    "CoHiRF-SC-SRGF-1R-60": "CoHiRF-SC-SRGF",
    "DBSCAN-60": "DBSCAN",
    "KMeans-60": "KMeans",
    "KernelRBFKMeans-60": "KernelRBFKMeans",
    "SpectralSubspaceRandomization-60": "SC-SRGF",
}

dataset_names = {
    "binary_alpha_digits": "binary-alpha-digits",
    "mnist_784": "mnist",
}  # otherwise we get an error in latex

dataset_id = [
    61,
    46773,
    46776,
    46778,
    46779,
    46782,
    46783,
    554,
    40685,
    1568,
    47039,
]

hpo_metrics = [
    "adjusted_rand",
    "calinski_harabasz_score",
    "silhouette",
]

# Filter to only standardized runs
df = df_runs_parents.copy()
df = df.loc[df["standardize"] == True]
df = df.loc[df["model"].isin(model_names.keys())]
df = df.loc[df["dataset_id"].isin(dataset_id)]
df = df.loc[df["hpo_metric"].isin(hpo_metrics)]
df = df.replace({"model": model_names})
df = df.replace({"dataset_name": dataset_names})

# Filter to only runs with hpo_seed in range(5)
df = df.loc[df["hpo_seed"].isin(range(5))]

# Filter to only show batch methods for datasets with more than 1000 instances
df = df.loc[~((df["n_instances"] < 1000) & (df["model"].str.find("Batch") != -1))]

# define group of models
model_groups = {
    "KMeans": [
        "KMeans",
        "CoHiRF",
        "R-CoHiRF",
        "CoHiRF-1000",
        "BatchCoHiRF",
        "R-BatchCoHiRF",
        "BatchCoHiRF-nolaststop",
        "R-BatchCoHiRF-nolaststop",
        "KernelRBFKMeans",
        "CoHiRF-KernelRBF",
        "R-CoHiRF-KernelRBF",
        "BatchCoHiRF-KernelRBF",
        "R-BatchCoHiRF-KernelRBF",
        "BatchCoHiRF-KernelRBF-nolaststop",
        "R-BatchCoHiRF-KernelRBF-nolaststop",
    ],
    "DBSCAN": [
        "DBSCAN",
        "CoHiRF-DBSCAN",
        "R-CoHiRF-DBSCAN",
        "BatchCoHiRF-DBSCAN",
        "R-BatchCoHiRF-DBSCAN",
        "BatchCoHiRF-DBSCAN-nolaststop",
        "R-BatchCoHiRF-DBSCAN-nolaststop",
    ],
    "SC-SRGF": [
        "SC-SRGF",
        "CoHiRF-SC-SRGF",
        "CoHiRF-SC-SRGF-1R",
        "CoHiRF-SC-SRGF-2R",
        "BatchCoHiRF-SC-SRGF",
        "BatchCoHiRF-SC-SRGF-2R",
        "BatchCoHiRF-SC-SRGF-nolaststop",
        "BatchCoHiRF-SC-SRGF-2R-nolaststop",
    ],
}
df["model_group"] = df["model"].apply(
    lambda x: next((group for group, models in model_groups.items() if x in models), "Other")
)


# re-scale some metrics and build composite metric
# re-scale ari to be between 0 and 1 (originally between -0.5 and 1), by considering everything below 0 as 0
df["best/adjusted_rand_rescaled"] = df["best/adjusted_rand"].apply(lambda x: 0.0 if x < 0 else x)

# re-scale silhouette to be between 0 and 1 (originally between -1 and 1)
df["best/silhouette_rescaled"] = (df["best/silhouette"] - (-1)) / (1 - (-1))

# re-scale calinski to be between 0 and 1 normalized by dataset, model_group and hpo_metric
# replace calinksi -1.0 with 0.0
df["best/calinski_harabasz_score"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
# df["best/calinski_harabasz_score_rescaled"] = df["best/calinski_harabasz_score"].replace(-1.0, 0.0)
df["best/calinski_harabasz_score_rescaled"] = df.groupby(["dataset_id", "model_group", "hpo_metric"])[
    "best/calinski_harabasz_score"
].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else (0.0 if x.max() == 0 else 1.0))

In [ ]:
hpo_metrics = [
    "adjusted_rand",
    "adjusted_rand_rescaled",
    "calinski_harabasz_score",
    "calinski_harabasz_score_rescaled",
    "silhouette",
    "silhouette_rescaled",
]

hpo_metrics_rename = [
    "ARI",
    "Rescaled ARI",
    "Calinski",
    "Rescaled Calinski",
    "Silhouette",
    "Rescaled Silhouette",
]

dfs_metrics = {}

for hpo_metric, hpo_metric_rename in zip(hpo_metrics, hpo_metrics_rename):
    if hpo_metric.find("_rescaled") != -1:
        original_metric = hpo_metric.replace("_rescaled", "")
    else:
        original_metric = hpo_metric
    df_metric = df.loc[df["hpo_metric"] == original_metric][
        ["dataset_name", "model", "hpo_seed", f"best/{hpo_metric}"]
    ].rename(columns={f"best/{hpo_metric}": hpo_metric_rename})
    df_metric = df_metric.dropna(subset=[hpo_metric_rename])
    df_metric = df_metric.set_index(["dataset_name", "model", "hpo_seed"])
    df_metric = df_metric.astype({hpo_metric_rename: float})
    dfs_metrics[hpo_metric_rename] = df_metric

df_metrics = pd.concat(dfs_metrics.values(), axis=1, join="outer")
df_metrics = df_metrics.reset_index()

# calculate mean and std
df_metrics = df_metrics.groupby(["dataset_name", "model"]).agg(["mean", "std"])
# flatten multiindex columns
df_metrics.columns = [" ".join(col).strip() for col in df_metrics.columns.values]
# drop hpo_seed level
df_metrics = df_metrics.drop(columns=["hpo_seed mean", "hpo_seed std"])
# Rename index levels
df_metrics.index.names = ["Dataset", "Model"]

# create a composite metric as the average of the metrics
df_metrics["Composite Metric mean"] = df_metrics[
    [f"{metric} mean" for metric in hpo_metrics_rename if "Rescaled" in metric]
].mean(axis=1)
df_metrics["Composite Metric std"] = (
    1
    / len(hpo_metrics_rename)
    * (df_metrics[[f"{metric} std" for metric in hpo_metrics_rename if "Rescaled" in metric]] ** 2).sum(axis=1) ** 0.5
)
hpo_metrics_rename.append("Composite Metric")


for metric in hpo_metrics_rename:
    df_metrics[f"{metric}"] = (
        df_metrics[f"{metric} mean"].apply(lambda x: f"{x:.3f}" if not pd.isna(x) else "No Run")
        + " $\\pm$ "
        + df_metrics[f"{metric} std"].apply(lambda x: f"{x:.3f}" if not pd.isna(x) else "No Run")
    )

In [ ]:
df_times = (
# Calculate mean and std times for each dataset-model combination across all metrics
    df.groupby(["dataset_name", "model"])
    .agg({"best/elapsed_time": ["mean", "std"], "fit_model_return_elapsed_time": ["mean", "std"]})
    .rename(columns={"best/elapsed_time": "Best Time", "fit_model_return_elapsed_time": "HPO Time"})
)

# Flatten multiindex columns
df_times.columns = [" ".join(col).strip() for col in df_times.columns.values]
# Set the same index structure as df_metrics
df_times.index.names = ["Dataset", "Model"]

df_times["Best Time"] = (
    df_times["Best Time mean"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
    + " $\\pm$ "
    + df_times["Best Time std"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
)
df_times["HPO Time"] = (
    df_times["HPO Time mean"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
    + " $\\pm$ "
    + df_times["HPO Time std"].apply(lambda x: f"{x:4.3f}" if not pd.isna(x) else "No Run")
)

# Join with the existing df_metrics (verify we have the same number of rows!)
df_metrics = df_metrics.join(df_times, how="outer")

In [ ]:
df_latex = df_metrics.copy()
df_latex = df_latex.reset_index()
# reapply model groups
df_latex["Base Model"] = df_latex["Model"].apply(
    lambda x: next((group for group, models in model_groups.items() if x in models), "Other")
)
df_latex = df_latex.loc[df_latex["Base Model"] == "KMeans"]
df_latex = df_latex.loc[df_latex["Dataset"] == "iris"]
df_latex = df_latex.loc[df_latex["Model"] != "CoHiRF-1000"]
# redefine index with model_group
df_latex = df_latex.set_index(["Dataset", "Base Model", "Model"])
# sort by dataset, model_group, model
df_latex = df_latex.sort_index(level=["Dataset", "Base Model", "Model"])


# print per dataset
for dataset in df_latex.index.get_level_values("Dataset").unique():
    df_print = df_latex.copy()
    df_print = df_print.loc[dataset]
    hpo_metrics_to_hide = [metric for metric in hpo_metrics_rename if metric.find("Rescaled") != -1]
    columns_to_hide = [col for col in df_latex.columns if col not in (hpo_metrics_rename)]
    columns_to_hide += hpo_metrics_to_hide
    df_print = df_print.style.hide(columns_to_hide, axis=1)
    for col in hpo_metrics_rename:
        highlight_metric = partial(highlight_max, column_name=f"{col} mean")
        underline_2nd_metric = partial(underline_2nd_max, column_name=f"{col} mean")
        if col in ["Davies-Bouldin", "Best Time", "HPO Time"]:
            highlight_metric = partial(highlight_min, column_name=f"{col} mean")
            underline_2nd_metric = partial(underline_2nd_min, column_name=f"{col} mean")
        (
            df_print.apply(highlight_metric, subset=[col, f"{col} mean"], axis=None).apply(
                underline_2nd_metric, subset=[col, f"{col} mean"], axis=None
            )
        )

    df_print = df_print.hide(level=0, axis=0)
    latex_output = df_print.to_latex(
        hrules=True,
        clines="skip-last;data",
        convert_css=True,
        column_format="ll" + "l" * (len(df_print.columns) - len(columns_to_hide)),
        # environment="longtable",
        caption=f"Clustering results on dataset {dataset}",
    )

    # fix header
    columns = df_print.index.names[1:] + [col for col in df_print.columns if col not in columns_to_hide]
    header_line = " & ".join(columns) + r" \\"

    # split into lines
    latex_output = latex_output.splitlines()
    # remove 5th and 6th line and replace with header_line
    latex_output = latex_output[:4] + [header_line] + latex_output[6:]
    latex_output = "\n".join(latex_output)

    # manually add clines after model groups
    model_groups_in_data = df_print.index.get_level_values("Base Model").unique().tolist()
    lines = latex_output.splitlines()
    new_lines = []
    last_line = ""
    for i, line in enumerate(lines[6:-3]):  # skip first 6 lines and last 3 lines
        model_last_line = last_line.split("&")[0].strip()
        model_current_line = line.split("&")[0].strip()
        model_group_last_line = next(
            (group for group, models in model_groups.items() if model_last_line in models), "Other"
        )
        model_group_current_line = next(
            (group for group, models in model_groups.items() if model_current_line in models), "Other"
        )
        if model_group_last_line != model_group_current_line and i != 0:
            new_lines.append(r"\cline{" + f"1-{len(columns)}" + r"}")
        new_lines.append(line)
        last_line = line

    latex_output = "\n".join(lines[:6] + new_lines + lines[-3:])

    print(latex_output)
    print("\n\n")